In [0]:
%sql
SELECT
    AVG(p.population) AS mean,
    STDDEV(p.population) AS stddev
FROM workspace.gold.population p
INNER JOIN workspace.gold.country c
    ON p.nation_id = c.nation_id
WHERE p.year BETWEEN 2013 AND 2018
  AND c.nation = 'United States';

In [0]:
%sql
WITH annual AS (
    SELECT
        series_id,
        year,
        SUM(value) AS annual_sum
    FROM workspace.gold.data
    WHERE quarter IN ('Q01', 'Q02', 'Q03', 'Q04')
    GROUP BY series_id, year
),

ranked AS (
    SELECT
        series_id,
        year,
        annual_sum,
        ROW_NUMBER() OVER (
            PARTITION BY series_id
            ORDER BY annual_sum DESC
        ) AS rn
    FROM annual
),

class_mapping AS (
    SELECT DISTINCT
        series_id,
        class_code
    FROM workspace.gold.data
)

SELECT
    r.series_id,
    r.year,
    r.annual_sum,
    c.* EXCEPT (class_code)
FROM ranked r
INNER JOIN class_mapping m
    ON r.series_id = m.series_id
INNER JOIN workspace.gold.class c
    ON m.class_code = c.class_code
WHERE r.rn = 1;

In [0]:
%sql
SELECT
    d.year,
    d.quarter,
    d.value,
    p.Population AS population
FROM workspace.gold.data d
LEFT JOIN workspace.gold.population p
    ON CAST(d.year AS INT) = CAST(p.Year AS INT)
WHERE TRIM(d.series_id) = 'PRS88003191'
  AND d.quarter = 'Q01'
ORDER BY d.year;